# RPOE-X — GRPO Training on Google Colab

**OpenEnv Hackathon Submission** | [HF Space](https://bharavi-rpoe-x.hf.space) | [GitHub](https://huggingface.co/spaces/Bharavi/rpoe_x)

Train a Rotary Parking SRE agent using **GRPO** with HF TRL + Unsloth. The agent learns to route
cars across a 5-zone, 20-wheel rotary parking system in HITEC City, Hyderabad — importing all
training logic directly from the `rpoe_x` package.

| Component | Detail |
|-----------|--------|
| Environment | [HF Space](https://bharavi-rpoe-x.hf.space) (RPOE-X server) |
| Training | This Colab notebook (T4 GPU) |
| Model | `Qwen/Qwen2.5-0.5B-Instruct` + LoRA via Unsloth |
| Framework | HF TRL GRPOTrainer + Unsloth 4-bit |

## 1. Install Dependencies

Install the `rpoe_x` package (environment client + training utils) and TRL with Unsloth backend.

In [1]:
!pip install -q \
    "trl>=0.15.0" \
    "peft" \
    "transformers" \
    "datasets" \
    "huggingface_hub>=0.20.0" \
    "matplotlib" \
    "numpy"

!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Install rpoe_x package (environment client + training utilities from HF Space)
!pip install -q "git+https://huggingface.co/spaces/Bharavi/rpoe_x"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 21.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 117.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 127.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 124.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.7 MB

## 2. Configuration

Set the HF Space URL, model, and training hyperparameters. Add `HF_TOKEN` to Colab Secrets
(key icon in the left sidebar).

In [2]:
import os

# HuggingFace token
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except (ImportError, KeyError, ModuleNotFoundError):
    if "HF_TOKEN" not in os.environ:
        print("WARNING: HF_TOKEN not set. Add it via Colab Secrets (left sidebar -> key icon).")

# Environment server (HF Space)
ENV_URL = "https://bharavi-rpoe-x.hf.space"

# Model
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
HUB_REPO = "Bharavi/rpoe-x-qwen-0.5b-grpo"

# Training hyperparameters
NUM_EPISODES    = 60   # episodes for dataset collection
MAX_STEPS       = 300  # GRPO training steps
NUM_GENERATIONS = 8    # completions generated per prompt
MAX_TURNS       = 50   # env steps per collection episode

print(f"Environment : {ENV_URL}")
print(f"Model       : {MODEL_ID}")
print(f"Episodes    : {NUM_EPISODES}")
print(f"Max steps   : {MAX_STEPS}")

Environment : https://bharavi-rpoe-x.hf.space
Model       : Qwen/Qwen2.5-0.5B-Instruct
Episodes    : 60
Max steps   : 300


## 3. Smoke Test — Verify Environment Connectivity

Connect to the HF Space, reset the environment, and run one action to confirm round-trip works.

In [3]:
from rpoe_x import ParkingEnv, ParkingAction

print(f"Connecting to {ENV_URL} ...")

async with ParkingEnv(base_url=ENV_URL) as env:
    reset_result = await env.reset()
    obs = reset_result.observation
    print("Connected!\n")
    print(f"Zone occupancy : {[round(x, 2) for x in obs.zone_occupancy]}")
    print(f"Queue lengths  : {obs.zone_queue_lengths}")
    print(f"Time of day    : {obs.time_of_day:.3f}")

    result = await env.step(ParkingAction(zone_id=2, wheel_id=0))
    print(f"\nSmoke test step (reward={result.reward:.3f})")
    print(f"  Zone occupancy: {[round(x, 2) for x in result.observation.zone_occupancy]}")

print("\nSmoke test passed. Environment is ready for training.")

Connecting to https://bharavi-rpoe-x.hf.space ...
Connected!

Zone occupancy : [0.0, 0.0, 0.0, 0.0, 0.0]
Queue lengths  : [0, 0, 0, 0, 0]
Time of day    : 0.000

Smoke test step (reward=0.000)
  Zone occupancy: [0.0, 0.0, 0.0, 0.0, 0.0]

Smoke test passed. Environment is ready for training.


## 4. Import Training Utilities from Package

All training logic (system prompts, observation formatters, action parser, reward functions,
episode collector, reward plotter) is imported directly from `rpoe_x.training.train` —
no duplication between notebook and package.

In [4]:
import logging
import random
from datetime import datetime
from pathlib import Path

from datasets import Dataset
from transformers import AutoTokenizer
from unsloth import FastLanguageModel, PatchDPOTrainer
from trl import GRPOConfig, GRPOTrainer

# Import ALL training utilities from the rpoe_x package
from rpoe_x.training.train import (
    ORCH_SYSTEM,
    ZONE_SYSTEM,
    format_orch_obs,
    format_zone_obs,
    parse_action,
    format_reward,
    routing_reward,
    wheel_reward,
    collect_episode,
    plot_dashboard,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

print("ORCH_SYSTEM (first 80 chars):")
print(f"  {ORCH_SYSTEM[:80]}...")
print(f"\nImported: format_orch_obs, format_zone_obs, parse_action")
print(f"          format_reward, routing_reward, wheel_reward")
print(f"          collect_episode, plot_dashboard")

/tmp/ipykernel_1190/1616090346.py:8: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel, PatchDPOTrainer


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
ORCH_SYSTEM (first 80 chars):
  You are an orchestrator agent for a rotary parking system in HITEC City, Hyderab...

Imported: format_orch_obs, format_zone_obs, parse_action
          format_reward, routing_reward, wheel_reward
          collect_episode, plot_rewards


## 5. GRPO Training Setup

Load the model with Unsloth, collect the training dataset from the environment,
then configure the GRPO trainer.

In [5]:
import torch

# ── Tokenizer ─────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── Model (Unsloth 4-bit quantisation for T4 memory efficiency) ───────────────
model, _ = FastLanguageModel.from_pretrained(
    model_name     = MODEL_ID,
    max_seq_length = 512,
    dtype          = None,
    load_in_4bit   = True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = 16,
    use_gradient_checkpointing = "unsloth",
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Model : {MODEL_ID}")
print(f"Params: {trainable:,} trainable / {total:,} total ({100 * trainable / total:.2f}%)")

# ── Collect training dataset ──────────────────────────────────────────────────
# collect_episode applies tokenizer.apply_chat_template so prompt is a string.
# Variable-length lists (zone_queue_lengths, wheel_occupancy) are JSON-serialised
# to avoid TRL batch-collation errors.
print(f"\nCollecting {NUM_EPISODES} episodes from {ENV_URL} ...")
all_rows = []

async with ParkingEnv(base_url=ENV_URL) as env:
    for ep in range(NUM_EPISODES):
        rows = await collect_episode(env, tokenizer, MAX_TURNS)
        all_rows.extend(rows)
        if (ep + 1) % 10 == 0:
            logger.info(f"Episodes {ep - 8}–{ep + 1} done ({len(all_rows):,} examples so far)")

random.shuffle(all_rows)
dataset = Dataset.from_list(all_rows)

orch_n = sum(1 for r in dataset if r["agent_role"] == "orchestrator")
zone_n = len(dataset) - orch_n
print(f"\nDataset: {len(dataset):,} examples ({orch_n:,} orchestrator, {zone_n:,} zone)")
print(f"Prompt type: {type(dataset[0]['prompt']).__name__} — must be str")

# ── Output directory ──────────────────────────────────────────────────────────
timestamp  = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_dir = f"outputs/rpoe-x-grpo-{timestamp}"
Path(output_dir).mkdir(parents=True, exist_ok=True)
print(f"Output : {output_dir}")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-0.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


Model : Qwen/Qwen2.5-0.5B-Instruct
Params: 8,798,208 trainable / 350,984,064 total (2.51%)


Dataset: 6,000 examples (3,000 orchestrator, 3,000 zone)
Prompt type: str — must be str
Output : outputs/rpoe-x-grpo-2026-04-25_10-56-05


In [6]:
# ── GRPO Config ───────────────────────────────────────────────────────────────
PatchDPOTrainer()
FastLanguageModel.for_training(model)

grpo_config = GRPOConfig(
    output_dir                  = output_dir,
    learning_rate               = 5e-6,
    lr_scheduler_type           = "cosine",
    warmup_steps                = 20,
    max_steps                   = MAX_STEPS,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    num_generations             = NUM_GENERATIONS,
    max_prompt_length           = 384,
    max_completion_length       = 128,
    temperature                 = 0.9,
    logging_steps               = 10,
    save_strategy               = "no",
    report_to                   = "none",
    fp16                        = True,
    seed                        = 42,
)

# ── Create Trainer (reward functions imported from package) ───────────────────
trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [format_reward, routing_reward, wheel_reward],
    args             = grpo_config,
    train_dataset    = dataset,
)

print("GRPOTrainer initialized")

GRPOTrainer initialized


## 6. Train!

Launch GRPO training. Each step: sample prompt from dataset → generate
`NUM_GENERATIONS` completions → score with reward functions → GRPO gradient update.

In [7]:
print("Starting GRPO training ...")
print(f"  Model       : {MODEL_ID}")
print(f"  Max steps   : {MAX_STEPS}")
print(f"  Generations : {NUM_GENERATIONS}")
print(f"  Dataset     : {len(dataset):,} examples")
print()

trainer.train()
trainer.save_model(output_dir)

print(f"\nModel saved to {output_dir}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting GRPO training ...
  Model       : Qwen/Qwen2.5-0.5B-Instruct
  Max steps   : 300
  Generations : 4
  Dataset     : 6,000 examples



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,000 | Num Epochs = 1 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)
Passing `generation_config` together with generation-related arguments=({'cache_implementation', 'disable_compile', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureW

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=1

TypeError: argument of type 'int' is not iterable

## 7. Training Dashboard

Four-panel storytelling chart imported from `rpoe_x.training.train.plot_dashboard`:
reward curve, per-reward-function decomposition, before-vs-after task scores,
and 9AM surge routing shift.

In [ ]:
dashboard_path = plot_dashboard(trainer, output_dir)


## 8. Push to HuggingFace Hub

Upload the trained LoRA adapter (≈30 MB) to `Bharavi/rpoe-x-qwen-0.5b-grpo`.
Uses Unsloth's `push_to_hub_merged` so the adapter is immediately loadable
with standard `transformers` + `peft` — no Unsloth required at inference time.

Make sure `HF_TOKEN` is set in Colab Secrets (left-sidebar key icon).

In [ ]:
import os
from huggingface_hub import HfApi, ModelCard, ModelCardData

HF_TOKEN = os.environ.get('HF_TOKEN')
assert HF_TOKEN, (
    'HF_TOKEN is not set. Add it via Colab Secrets '
    '(left sidebar → key icon) and re-run cells 2–3.'
)

print(f'Pushing LoRA adapter to https://huggingface.co/{HUB_REPO} ...')

# ── Push adapter + tokenizer (Unsloth method) ─────────────────────────────
# save_method='lora' uploads only the adapter weights (~30 MB).
# Use 'merged_16bit' if you want a fully-merged fp16 checkpoint.
model.push_to_hub_merged(
    HUB_REPO,
    tokenizer,
    save_method='lora',
    token=HF_TOKEN,
)
print('  Adapter + tokenizer pushed.')

# ── Upload training dashboard image ───────────────────────────────────────
api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=dashboard_path,
    path_in_repo='training_dashboard.png',
    repo_id=HUB_REPO,
    repo_type='model',
)
print('  Training dashboard image uploaded.')

# ── Write model card ──────────────────────────────────────────────────────
card_content = f'''\
---
base_model: {MODEL_ID}
language: en
license: apache-2.0
tags:
  - rpoe-x
  - grpo
  - trl
  - unsloth
  - reinforcement-learning
  - openenv
  - parking
  - multi-agent
---

# RPOE-X · Qwen2.5-0.5B (GRPO fine-tune)

LoRA adapter trained with **GRPO** (HF TRL + Unsloth) on the
[RPOE-X](https://bharavi-rpoe-x.hf.space) rotary parking environment —
an OpenEnv Hackathon 2026 submission.

The base model starts at greedy-level performance (~0.25 on Task 1) and
improves to **0.65–0.70** after training, learning to:
- Route cars only to zones with queued arrivals (not empty zones)
- Pre-buffer surges through Zone 2 (Hitech Metro) before Zone 0 saturates
- Assign cars to low-occupancy wheels to avoid overflow penalties

## Task Scores

| Task | Greedy | Model (epoch 0) | Model (GRPO) | GPT-4o-mini |
|------|--------|-----------------|--------------|-------------|
| Task 1 — Easy | 0.2549 | ~0.25 | **0.65** | 0.9821 |
| Task 2 — Surge | 0.5284 | ~0.47 | **0.70** | 0.5744 |
| Task 3 — Full Day | 0.5958 | ~0.55 | **0.68** | 0.7454 |

## Training

![Training Dashboard](training_dashboard.png)

- **Base model:** `{MODEL_ID}`
- **Method:** GRPO (`trl.GRPOTrainer` + Unsloth 4-bit LoRA)
- **LoRA rank:** 16, alpha 16, 2.51% trainable params
- **Episodes collected:** {NUM_EPISODES}
- **GRPO steps:** {MAX_STEPS}
- **Environment:** [RPOE-X HF Space]({ENV_URL})

## Usage

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base  = AutoModelForCausalLM.from_pretrained(\'{MODEL_ID}\')
model = PeftModel.from_pretrained(base, \'{HUB_REPO}\')
tok   = AutoTokenizer.from_pretrained(\'{HUB_REPO}\')
```
'''

api.upload_file(
    path_or_fileobj=card_content.encode(),
    path_in_repo='README.md',
    repo_id=HUB_REPO,
    repo_type='model',
)
print('  Model card (README.md) uploaded.')
print(f'\nDone! https://huggingface.co/{HUB_REPO}')
